# 30. `ai_summary` 로 표현↔accord 신호를 측정한다

| | |
|---|---|
| 질문 | 사람들이 특정 표현으로 부른 향수들이 **실제로 어떤 accord 를 갖는가** |
| 왜 | 사전 매핑 28행이 전부 `status=candidate` 다. 의미 판정 근거가 없다 |
| 입력 | `perfumes.jsonl` 의 `ai_summary` — **이번에 처음 쓰는 필드** |
| API | **호출하지 않는다** |
| 작성 | 2026-09-11 |

## 왜 이 필드인가

`SCHEMA.md` — *"`ai_summary.pros/cons` | Fragrantica의 AI-요약 의견;
`up_votes`/`down_votes` = 동의/비동의한 사용자 수"*

즉 **향수별 서술문 + 몇 명이 동의했는가** 다. 실물은 이렇게 생겼다.

```
Givenchy / Amarige
  +142 -11   Creamy luscious tuberose
   +98 -36   Potentially overpowering and loud projection
```

지금까지 두 번의 외부 조사(전문가 문헌 / 국내 커뮤니티)가 찾으려던 것이 정확히 이것이다.
전문가 조사는 개별 표현 근거 **0건**, 커뮤니티 조사는 **글 1~3건**을 얻었다.
이 필드는 같은 성격의 근거를 **수만 건** 규모로 갖고 있다.

**`perfumes.csv` 에는 이 필드가 없다.** 59개 컬럼 중 서술 관련은 `description` 하나뿐이고
`ai_summary` 는 jsonl 에만 있다. EDA 01~29 가 전부 CSV 로 작업해 보이지 않았다.

## 이 노트북이 답하지 않는 것

- **전문 조향사·공식 기관 근거가 아니다.** Fragrantica 의 AI 가 사용자 리뷰를 요약한 것이다
- **한국어 표현이 아니다.** 영어 서술이므로 `포근한 = cozy` 라는 번역 가정이 한 단계 들어간다
- **사전을 고치지 않는다.** 측정만 하고 반영 여부는 사람이 정한다

## 0. 실행 조건과 한계

### 사전 등록이 훼손된 부분을 먼저 밝힌다

**나는 이 노트북을 쓰기 전에 6개 표현의 결과를 이미 봤다** — `포근한`·`빨래`·`깨끗한`·
`휴양지`·`무난한`·`고급스러운`. 대화 중 탐색으로 돌린 것이다.

그래서 두 가지를 지킨다.

1. **탐색 때 쓴 영어 대응어를 바꾸지 않는다.** 신호가 잘 나오는 단어로 갈아끼우면
   그건 측정이 아니라 조작이다
2. **본 적 없는 표현도 전부 포함한다.** 6개만 보고하면 선택 편향이 된다.
   사전의 ACCORD 매핑 표현 전부와 질감층까지 넣는다

### 한계 — 결과를 읽을 때 반드시 함께 볼 것

- **출처가 Fragrantica 다.** `spec.md` §8 의 `robots.txt ai-train=no` · EU 권리유보 문제가
  그대로 적용된다. 이 필드를 서비스에 쓰려면 그 판단이 선행돼야 한다
- **커버리지가 좁다.** 전체 131,930건 중 `ai_summary` 보유는 일부다. §3 에서 **편향을 측정**한다
- **영어 대응어는 내 판단이다.** 다만 `포근한 = cozy` 는 **일반 번역**이지 향 도메인 추론이 아니다.
  `포근한 → powdery` 같은 향 특화 판단보다 가정이 약하다
- **검색어가 accord 이름과 같으면 순환이다.** `tropical`·`sweet`·`musky` 가 그렇다.
  §2 에서 자동 검출해 표시하고, 해당 accord 의 배수는 신호로 읽지 않는다
- **부분 문자열 검색이다.** `light` 가 `lighthearted` 에, `natural` 이 `natural ingredients` 에
  걸린다. 잡음을 줄이지 못한 채로 보고한다
- **투표수를 가중치로 쓰지 않는다.** 이번에는 서술문의 **존재 여부**만 본다

### 하지 않는 것

사전 수정, `spec.md` 스키마 변경, 평가 데이터 수정, 기존 노트북 수정·실행, LLM 호출.

In [1]:
import hashlib
import json
import pathlib
import random

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

print("REPORT_ONLY:", REPORT_ONLY)

REPORT_ONLY: False


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"

INPUT_PATHS = {
    "perfumes_jsonl": PROJECT_ROOT / "perfumes.jsonl",
    "accord_dictionary": OUTPUT_DIR / "10_accord_dictionary.csv",
    "lexicon": KNOWLEDGE_DIR / "domain_lexicon_v1.csv",
}
OUTPUT_PATHS = {
    "lift": OUTPUT_DIR / "30_expression_accord_lift.csv",
    "crosscheck": OUTPUT_DIR / "30_lexicon_crosscheck.csv",
    "summary": OUTPUT_DIR / "30_ai_summary_signal_report.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {
    (KNOWLEDGE_DIR / "domain_lexicon_v1.csv").resolve(),
    (KNOWLEDGE_DIR / "community_product_alias_v1.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "semantic_bridge"
     / "22_pilot_human_evaluation.csv").resolve(),
    (PROJECT_ROOT / "perfumes.jsonl").resolve(),
    (PROJECT_ROOT / "perfumes.csv").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 의 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
perfumes_jsonl,414478c1921f30dd
accord_dictionary,567e91370e575731
lexicon,936808a9b3c32c2b


## 2. 사전 등록 — 데이터를 보기 전에 고정한다

영어 대응어와 판정 규칙을 데이터 로드보다 앞에 둔다.

`pre_seen` 이 `True` 인 6개는 **내가 이미 결과를 본 표현**이다. 단어를 바꾸지 않았다는 점을
이 플래그로 추적할 수 있게 표시해 둔다.

In [3]:
PREREG = {
    "notebook": "30_ai_summary_expression_accord",
    "question": "특정 표현으로 불린 향수들이 실제로 어떤 accord 를 갖는가",
    "llm_calls": 0,
    "source_field": "perfumes.jsonl :: ai_summary.pros[].text + cons[].text",
    "match_rule": "소문자 부분 문자열 포함. 투표수 가중 없음",

    # 판정 기준
    "min_support": 30,        # 해당군에서 이 개수 미만인 accord 는 버린다
    "min_subset": 100,        # 표현에 걸린 향수가 이 개수 미만이면 판정하지 않는다
    "signal_lift": 1.5,       # 배수가 이 값 이상이고 CI 하한이 1 초과면 신호로 본다
    "control_seed": 42,       # 통제군 난수 시드

    "known_deviation": (
        "6개 표현(포근한·빨래·깨끗한·휴양지·무난한·고급스러운)은 이 노트북 작성 전에 "
        "탐색으로 결과를 봤다. 그때 쓴 영어 대응어를 바꾸지 않았고, 보지 않은 표현도 전부 포함했다."
    ),
}

# 표현 → 영어 대응어. accord 이름과 겹치는 것은 §2 끝에서 자동 검출한다.
EXPRESSION_TERMS = {
    # ── 사전의 ACCORD 매핑 표현
    "포근한":     (["cozy", "cosy", "comforting", "snuggl"], True),
    "깨끗한":     (["clean", "crisp"], True),
    "빨래":       (["laundry", "linen", "detergent", "fabric softener", "freshly washed"], True),
    "이불":       (["bedding", "cotton", "bed sheet", "blanket"], False),
    "비 오는 숲":  (["petrichor", "wet earth", "after rain", "damp forest", "rainy"], False),
    "차가운":     (["cold", "icy", "chilly"], False),
    "촉촉한":     (["dewy", "watery", "moist"], False),
    "휴양지":     (["beach", "vacation", "holiday", "tropical", "resort"], True),
    "호텔":       (["hotel", "spa", "amenity"], False),
    "머스크":     (["white musk", "clean musk", "musky"], False),
    "달달":       (["sweet", "sugary", "candy"], False),
    # ── 사전이 NO_MAPPING 으로 둔 표현
    "무난한":     (["versatile", "easy to wear", "crowd-pleas", "inoffensive"], True),
    "고급스러운":  (["elegant", "luxurious", "sophisticat", "classy"], True),
    "도시적인":   (["urban", "modern"], False),
    "섹시한":     (["sexy", "seductive", "sensual"], False),
    "자연스러운":  (["natural"], False),
    "향긋한":     (["fragrant"], False),
    # ── spec.md §8 남은 작업 5번(질감층). 미결정이므로 측정만 한다
    "무거운":     (["heavy", "cloying"], False),
    "가벼운":     (["light", "airy"], False),
}

MIN_SUPPORT = PREREG["min_support"]
MIN_SUBSET = PREREG["min_subset"]
SIGNAL_LIFT = PREREG["signal_lift"]

print(f"표현 {len(EXPRESSION_TERMS)}개 / 이미 본 것 "
      f"{sum(1 for _, seen in EXPRESSION_TERMS.values() if seen)}개")
display(Markdown("```json\n" + json.dumps(PREREG, ensure_ascii=False, indent=2) + "\n```"))

표현 19개 / 이미 본 것 6개


```json
{
  "notebook": "30_ai_summary_expression_accord",
  "question": "특정 표현으로 불린 향수들이 실제로 어떤 accord 를 갖는가",
  "llm_calls": 0,
  "source_field": "perfumes.jsonl :: ai_summary.pros[].text + cons[].text",
  "match_rule": "소문자 부분 문자열 포함. 투표수 가중 없음",
  "min_support": 30,
  "min_subset": 100,
  "signal_lift": 1.5,
  "control_seed": 42,
  "known_deviation": "6개 표현(포근한·빨래·깨끗한·휴양지·무난한·고급스러운)은 이 노트북 작성 전에 탐색으로 결과를 봤다. 그때 쓴 영어 대응어를 바꾸지 않았고, 보지 않은 표현도 전부 포함했다."
}
```

### 순환 검사 — 검색어가 accord 이름과 같은가

`tropical` 로 검색해서 `tropical` accord 가 나오면 그건 신호가 아니라 동어반복이다.
미리 찾아 표시하고, 결과 표에서 `순환` 으로 마킹한다.

In [4]:
accord_master = pd.read_csv(INPUT_PATHS["accord_dictionary"])
ACCORD_NAMES = set(accord_master.accord)

CIRCULAR = {}
for expr, (terms, _) in EXPRESSION_TERMS.items():
    same = sorted({a for a in ACCORD_NAMES for t in terms
                   if t.lower() == a.lower() or a.lower() in t.lower()})
    if same:
        CIRCULAR[expr] = same
for expr, names in CIRCULAR.items():
    print(f"  {expr:10s} 검색어가 accord 이름과 같음 → {names}")
print(f"\n순환 위험 표현 {len(CIRCULAR)}개 / 전체 {len(EXPRESSION_TERMS)}개")

  빨래         검색어가 accord 이름과 같음 → ['fresh']
  휴양지        검색어가 accord 이름과 같음 → ['tropical']
  머스크        검색어가 accord 이름과 같음 → ['musky']
  달달         검색어가 accord 이름과 같음 → ['sweet']

순환 위험 표현 4개 / 전체 19개


## 3. `perfumes.jsonl` 한 번 읽기

509MB 를 한 번만 훑으면서 필요한 것만 뽑는다. 원본은 읽기만 한다.

- accord 이름 집합
- `ai_summary` 서술문을 이어붙인 소문자 문자열 → 표현별 포함 여부
- 커버리지 편향 측정용 지표 (`people`, `popularity.magnitude`, `year`)

In [5]:
accords_list = []       # 향수별 accord 집합
has_summary = []        # ai_summary 보유 여부
expr_hits = {e: [] for e in EXPRESSION_TERMS}
people_list, pop_list, year_list = [], [], []

n_total = n_summary = n_stmt = 0
with INPUT_PATHS["perfumes_jsonl"].open(encoding="utf-8") as handle:
    for line in handle:
        rec = json.loads(line)
        n_total += 1
        acc = {a["name"] for a in (rec.get("accords") or []) if a.get("name")}
        accords_list.append(acc)

        summary = rec.get("ai_summary") or {}
        items = (summary.get("pros") or []) + (summary.get("cons") or [])
        blob = " ".join((i.get("text") or "") for i in items).lower()
        has_summary.append(bool(items))
        if items:
            n_summary += 1
            n_stmt += len(items)
        for expr, (terms, _) in EXPRESSION_TERMS.items():
            expr_hits[expr].append(bool(blob) and any(t in blob for t in terms))

        people_list.append(rec.get("people") or 0)
        pop_list.append(((rec.get("popularity") or {}).get("magnitude")) or 0)
        year_list.append(rec.get("year"))

has_summary = np.array(has_summary)
expr_hits = {e: np.array(v) for e, v in expr_hits.items()}
has_accord = np.array([bool(a) for a in accords_list])

print(f"향수 {n_total:,}건")
print(f"  accord 보유        {has_accord.sum():,}건")
print(f"  ai_summary 보유    {n_summary:,}건 ({n_summary / n_total:.1%})")
print(f"  서술문 총          {n_stmt:,}개")
print(f"  둘 다 보유         {(has_summary & has_accord).sum():,}건")

향수 131,930건
  accord 보유        129,161건
  ai_summary 보유    12,432건 (9.4%)
  서술문 총          210,521개
  둘 다 보유         12,411건


## 4. 재현 게이트

파싱이 기존 산출물과 어긋나면 이후 숫자를 믿을 수 없다.
`10_accord_dictionary.csv` 의 `perfume_count` 를 재현하는지 먼저 확인한다.

In [6]:
counts = {}
for acc in accords_list:
    for a in acc:
        counts[a] = counts.get(a, 0) + 1
ref = dict(zip(accord_master.accord, accord_master.perfume_count))
gate = [(a, ref[a], counts.get(a, 0)) for a in ref if counts.get(a, 0) != ref[a]]
if gate:
    for a, e, g in gate[:10]:
        print(f"  {a:16s} 기존 {e:>7,}  파싱 {g:>7,}")
    raise RuntimeError(f"재현 게이트 실패 — accord {len(gate)}개 불일치")
print(f"재현 게이트 통과 — accord {len(ref)}개 perfume_count 오차 0")

재현 게이트 통과 — accord 92개 perfume_count 오차 0


## 5. 커버리지 편향 — `ai_summary` 는 어떤 향수에 붙어 있는가

전체의 일부에만 있으므로, 그 일부가 어떤 향수인지 확인해야 결과를 해석할 수 있다.
**인기 향수에 쏠려 있다면 대중적인 향의 신호만 잡힌다.**

In [7]:
people = np.array(people_list, dtype=float)
pop = np.array(pop_list, dtype=float)
years = pd.Series(year_list, dtype="Float64")

rows = []
for label, mask in [("ai_summary 있음", has_summary), ("없음", ~has_summary)]:
    rows.append({
        "구분": label,
        "향수 수": int(mask.sum()),
        "평가자 수 중앙값": float(np.median(people[mask])),
        "평가자 수 평균": float(people[mask].mean()),
        "인기도 중앙값": float(np.median(pop[mask])),
        "출시연도 중앙값": float(years[mask].median()) if years[mask].notna().any() else float("nan"),
    })
bias_df = pd.DataFrame(rows)
display(bias_df)

med_with = np.median(people[has_summary])
med_without = np.median(people[~has_summary])
ratio = med_with / med_without if med_without else float("inf")
print(f"평가자 수 중앙값 비율: {ratio:,.0f}배")
print("→ ai_summary 는 평가자가 많은 향수에 붙는다. 대중적·인기 향수 쪽으로 쏠린 표본이다.")

,구분,향수 수,평가자 수 중앙값,평가자 수 평균,인기도 중앙값,출시연도 중앙값
0,ai_summary 있음,12432,628.0,1330.400499,3290.0,2019.0
1,없음,119498,9.0,40.009916,67.0,2020.0


평가자 수 중앙값 비율: 70배
→ ai_summary 는 평가자가 많은 향수에 붙는다. 대중적·인기 향수 쪽으로 쏠린 표본이다.


## 6. 통제군 — 이 방법이 신호를 만들어내지는 않는가

`spec.md` §7.3 의 조건 5(무작위 통제군)와 같은 취지다.
**표현과 무관하게 같은 크기로 무작위 추출한 집단**에서 배수를 계산한다.
방법이 건전하면 전부 1.0 근처여야 한다.

In [8]:
both = has_summary & has_accord
base_counts = {}
for acc, ok in zip(accords_list, both):
    if ok:
        for a in acc:
            base_counts[a] = base_counts.get(a, 0) + 1
BASE_N = int(both.sum())


def lift_table(mask, label):
    """mask 에 걸린 향수들의 accord 배수. DataFrame."""
    m = int(mask.sum())
    if m < MIN_SUBSET:
        return pd.DataFrame(), m
    sub = {}
    for acc, ok in zip(accords_list, mask):
        if ok:
            for a in acc:
                sub[a] = sub.get(a, 0) + 1
    out = []
    for a, c in sub.items():
        if c < MIN_SUPPORT:
            continue
        p_sub, p_base = c / m, base_counts.get(a, 0) / BASE_N
        if p_base == 0:
            continue
        se = (p_sub * (1 - p_sub) / m) ** 0.5
        out.append({
            "표현": label, "accord": a, "해당군 수": c, "해당군 비율": p_sub,
            "전체 비율": p_base, "배수": p_sub / p_base,
            "배수 CI하한": max(p_sub - 1.96 * se, 0) / p_base,
            "순환": a in CIRCULAR.get(label, []),
        })
    return pd.DataFrame(out).sort_values("배수", ascending=False), m


rng = random.Random(PREREG["control_seed"])
idx = [i for i, ok in enumerate(both) if ok]
ctrl_rows = []
for size in [500, 1000, 2500]:
    pick = set(rng.sample(idx, size))
    mask = np.array([i in pick for i in range(n_total)])
    t, m = lift_table(mask, f"통제군 n={size}")
    if len(t):
        ctrl_rows.append({"통제군": f"n={size}", "accord 수": len(t),
                          "최대 배수": t["배수"].max(), "최소 배수": t["배수"].min(),
                          "배수 1.5 이상": int((t["배수"] >= SIGNAL_LIFT).sum())})
ctrl_df = pd.DataFrame(ctrl_rows)
display(ctrl_df)
print("→ 무작위 집단에서는 배수가 1 근처에 모여야 한다. 1.5 이상이 거의 없으면 방법이 건전하다.")

,통제군,accord 수,최대 배수,최소 배수,배수 1.5 이상
0,n=500,29,1.169152,0.909971,0
1,n=1000,42,1.384397,0.813303,0
2,n=2500,52,1.138048,0.786440,0


→ 무작위 집단에서는 배수가 1 근처에 모여야 한다. 1.5 이상이 거의 없으면 방법이 건전하다.


## 7. 표현별 accord 배수

**배수**는 그 표현으로 불린 향수 중 해당 accord 보유 비율을, 전체 비율로 나눈 값이다.
2.0 이면 전체보다 2배 흔하다는 뜻이다.

`CI하한` 은 해당군 비율의 95% 신뢰구간 하한으로 계산한 배수다. 이것도 1을 넘어야 신호로 본다.

In [9]:
all_tables = []
subset_sizes = {}
for expr in EXPRESSION_TERMS:
    t, m = lift_table(expr_hits[expr] & both, expr)
    subset_sizes[expr] = m
    if len(t):
        all_tables.append(t)

lift_df = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
lift_df["신호"] = (lift_df["배수"] >= SIGNAL_LIFT) & (lift_df["배수 CI하한"] > 1.0) & (~lift_df["순환"])

print(f"표현 {len(EXPRESSION_TERMS)}개 / 판정 가능 {len([e for e in subset_sizes if subset_sizes[e] >= MIN_SUBSET])}개")
print(f"측정된 (표현, accord) 조합 {len(lift_df):,}건 / 신호 판정 {int(lift_df['신호'].sum()):,}건\n")
for expr in EXPRESSION_TERMS:
    m = subset_sizes[expr]
    if m < MIN_SUBSET:
        print(f"{expr:10s} 향수 {m:>6,}개  — 표본 부족으로 판정하지 않음")
        continue
    t = lift_df[lift_df["표현"] == expr].head(6)
    top = "  ".join(f"{r.accord}{'*' if r.순환 else ''} {r.배수:.2f}x"
                    for r in t.itertuples())
    print(f"{expr:10s} 향수 {m:>6,}개  {top}")
print("\n* 표시는 검색어가 accord 이름과 같아 순환인 항목이다.")

표현 19개 / 판정 가능 18개
측정된 (표현, accord) 조합 781건 / 신호 판정 65건

포근한        향수  2,756개  savory 3.72x  nutty 2.03x  cacao 2.01x  lactonic 2.00x  almond 1.81x  caramel 1.76x
깨끗한        향수  4,044개  soapy 2.18x  aquatic 1.77x  aldehydic 1.73x  ozonic 1.63x  fresh 1.61x  green 1.52x
빨래         향수    957개  soapy 4.50x  aldehydic 2.94x  aquatic 1.92x  ozonic 1.80x  fresh* 1.77x  musky 1.65x
이불         향수    112개  caramel 7.36x  powdery 1.57x  musky 1.45x  vanilla 1.34x  sweet 1.31x  fruity 1.22x
비 오는 숲     향수    241개  mossy 3.63x  ozonic 3.16x  earthy 3.04x  aquatic 3.03x  green 2.00x  aromatic 1.38x
차가운        향수  4,010개  cinnamon 1.74x  chocolate 1.52x  rum 1.51x  tobacco 1.51x  metallic 1.49x  whiskey 1.49x
촉촉한        향수    368개  aquatic 3.30x  ozonic 2.86x  rose 2.26x  green 2.12x  fresh 1.93x  floral 1.92x
휴양지        향수  1,211개  coconut 5.76x  tropical* 5.74x  salty 4.16x  marine 3.07x  lactonic 2.07x  yellow floral 2.05x
호텔         향수  1,163개  Champagne 6.96x  aldehydic 2.39x  fresh 1.52x  herb

머스크        향수    653개  musky* 2.19x  iris 1.46x  violet 1.43x  powdery 1.37x  floral 1.36x  rose 1.31x
달달         향수  6,169개  caramel 1.61x  nutty 1.47x  sour 1.42x  almond 1.41x  lactonic 1.41x  savory 1.40x
무난한        향수  6,217개  camphor 1.30x  bitter 1.27x  salty 1.27x  marine 1.25x  lavender 1.22x  mineral 1.18x
고급스러운      향수  6,966개  iris 1.38x  tuberose 1.27x  violet 1.24x  oud 1.22x  yellow floral 1.22x  leather 1.21x
도시적인       향수  1,611개  aldehydic 2.29x  mossy 1.93x  lavender 1.80x  earthy 1.60x  herbal 1.40x  iris 1.38x
섹시한        향수  1,645개  rum 1.73x  cherry 1.69x  coffee 1.58x  patchouli 1.54x  cinnamon 1.52x  tuberose 1.52x
자연스러운      향수  1,830개  conifer 2.39x  herbal 2.00x  green 1.69x  mossy 1.38x  fresh spicy 1.35x  salty 1.33x
향긋한        향수     18개  — 표본 부족으로 판정하지 않음
무거운        향수  5,666개  caramel 1.49x  nutty 1.38x  savory 1.37x  chocolate 1.35x  almond 1.34x  coconut 1.34x
가벼운        향수  2,721개  soapy 1.62x  aquatic 1.43x  ozonic 1.41x  aldehydic 1.34x  floral 1.30

## 8. 사전과 대조 — 우리 매핑이 지지되는가 반증되는가

`domain_lexicon_v1.csv` 의 ACCORD 행마다, 그 표현의 배수 순위에서 해당 accord 가
어디에 있는지 본다.

In [10]:
lex = pd.read_csv(INPUT_PATHS["lexicon"], keep_default_na=False, dtype=str)
lex_acc = lex[lex.candidate_type == "ACCORD"]

rows = []
for r in lex_acc.to_dict("records"):
    expr, acc = r["expression"], r["candidate_name"]
    m = subset_sizes.get(expr)
    if m is None:
        verdict, lift, rank, n_acc = "표현 미측정", None, None, None
    elif m < MIN_SUBSET:
        verdict, lift, rank, n_acc = "표본 부족", None, None, m
    else:
        t = lift_df[lift_df["표현"] == expr].reset_index(drop=True)
        hit = t[t.accord == acc]
        n_acc = m
        if len(hit) == 0:
            verdict, lift, rank = "반증 — 해당군에 지지도 없음", None, None
        else:
            lift = float(hit.iloc[0]["배수"])
            rank = int(hit.index[0]) + 1
            if bool(hit.iloc[0]["순환"]):
                verdict = "순환 — 신호로 읽지 않음"
            elif bool(hit.iloc[0]["신호"]):
                verdict = "지지"
            elif lift >= 1.0:
                verdict = "약함"
            else:
                verdict = "반증 — 전체보다 드물다"
    rows.append({
        "entry_id": r["entry_id"], "표현": expr, "accord": acc,
        "required": r["required"], "등급": r["evidence_tier"],
        "해당군 향수": n_acc, "배수": lift, "배수 순위": rank, "판정": verdict,
    })

cross_df = pd.DataFrame(rows)
display(cross_df)
print()
print(cross_df["판정"].value_counts().to_string())

,entry_id,표현,accord,required,등급,해당군 향수,배수,배수 순위,판정
0,kr.drift.musk,머스크,musky,core,VERIFIED,653,2.191723,1.0,순환 — 신호로 읽지 않음
1,kr.drift.musk,머스크,soapy,core,TEAM,653,NaN,NaN,반증 — 해당군에 지지도 없음
2,kr.drift.musk,머스크,fresh,optional,LLM,653,1.268350,8.0,약함
3,kr.ctx.clean,깨끗한,soapy,core,TEAM,4044,2.177994,1.0,지지
4,kr.ctx.clean,깨끗한,fresh,core,LLM,4044,1.610654,5.0,지지
5,kr.ctx.clean,깨끗한,aquatic,core,TEAM,4044,1.765247,2.0,지지
6,kr.ctx.clean,깨끗한,fresh,core,TEAM,4044,1.610654,5.0,지지
7,kr.scene.laundry,빨래,soapy,core,TEAM,957,4.497194,1.0,지지
8,kr.scene.laundry,빨래,fresh,core,TEAM,957,1.765685,5.0,순환 — 신호로 읽지 않음
9,kr.sens.cozy,포근한,powdery,core,TEAM,2756,1.345754,14.0,약함



판정
지지                  14
약함                   5
순환 — 신호로 읽지 않음       4
반증 — 해당군에 지지도 없음     3
반증 — 전체보다 드물다        2


## 9. 저장

In [11]:
def fmt(v, nd=2):
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{nd}f}"


top_lines = []
for expr in EXPRESSION_TERMS:
    m = subset_sizes[expr]
    if m < MIN_SUBSET:
        top_lines.append(f"| {expr} | {m:,} | — | 표본 부족 |")
        continue
    t = lift_df[lift_df["표현"] == expr].head(5)
    s = " · ".join(f"`{r.accord}`{'*' if r.순환 else ''} {r.배수:.2f}x" for r in t.itertuples())
    top_lines.append(f"| {expr} | {m:,} | {s} | |")
top_table = "\n".join(top_lines)

cross_lines = "\n".join(
    f"| `{r['entry_id']}` | {r['표현']} | `{r['accord']}` | {r['required']} | {r['등급']} | "
    f"{fmt(r['배수'])} | {r['판정']} |"
    for r in cross_df.to_dict("records"))
verdict_counts = cross_df["판정"].value_counts().to_dict()
ctrl_max = float(ctrl_df["최대 배수"].max()) if len(ctrl_df) else float("nan")

report = f"""# `ai_summary` 로 본 표현↔accord 신호

## 질문

사람들이 특정 표현으로 부른 향수들이 **실제로 어떤 accord 를 갖는가.**
사전 매핑 28행이 전부 `status=candidate` 이고 의미 판정 근거가 없어서 측정했다.

## 입력 — 이번에 처음 쓰는 필드

`perfumes.jsonl` 의 `ai_summary.pros/cons`. `SCHEMA.md` 에 따르면
*"Fragrantica 의 AI-요약 의견; `up_votes`/`down_votes` = 동의/비동의한 사용자 수"* 다.

| | |
|---|---:|
| 향수 | {n_total:,} |
| `ai_summary` 보유 | {n_summary:,} ({n_summary / n_total:.1%}) |
| 서술문 | {n_stmt:,} |
| accord 와 둘 다 보유 | {BASE_N:,} |

**`perfumes.csv` 에는 이 필드가 없다.** 59개 컬럼 중 서술 관련은 `description` 하나뿐이다.

## 커버리지 편향 — 먼저 읽어야 할 것

| 구분 | 향수 수 | 평가자 수 중앙값 | 인기도 중앙값 |
|---|---:|---:|---:|
| `ai_summary` 있음 | {int(has_summary.sum()):,} | {np.median(people[has_summary]):,.0f} | {np.median(pop[has_summary]):,.0f} |
| 없음 | {int((~has_summary).sum()):,} | {np.median(people[~has_summary]):,.0f} | {np.median(pop[~has_summary]):,.0f} |

**이 필드는 평가자가 많은 향수에 붙는다.** 따라서 여기서 나온 신호는
'대중적으로 많이 쓰이는 향수에서의 표현↔accord 관계' 이고, 롱테일에는 적용을 보증할 수 없다.

## 통제군 — 방법이 신호를 만들어내지 않는가

표현과 무관하게 같은 크기로 무작위 추출한 집단에서 같은 계산을 했다.

| 통제군 | accord 수 | 최대 배수 | 배수 {SIGNAL_LIFT} 이상 |
|---|---:|---:|---:|
{chr(10).join(f"| {r['통제군']} | {r['accord 수']} | {r['최대 배수']:.2f} | {r['배수 1.5 이상']} |" for r in ctrl_df.to_dict("records"))}

무작위 집단의 최대 배수가 {ctrl_max:.2f} 다. 아래 표현별 배수와 비교해 읽는다.

## 표현별 상위 accord

`*` 는 검색어가 accord 이름과 같아 순환인 항목이며 신호로 읽지 않는다.

| 표현 | 해당 향수 | 상위 accord (배수) | 비고 |
|---|---:|---|---|
{top_table}

## 사전 대조

| entry_id | 표현 | accord | required | 등급 | 배수 | 판정 |
|---|---|---|---|---|---:|---|
{cross_lines}

판정 집계: {verdict_counts}

## 한계

- **전문 조향사·공식 기관 근거가 아니다.** Fragrantica 의 AI 가 사용자 리뷰를 요약한 것이다.
  투표로 검증된 대중 의견이라는 점이 커뮤니티 글 1~2건보다 나을 뿐이다
- **출처가 Fragrantica 다.** `spec.md` §8 의 `ai-train=no` · EU 권리유보 문제가 그대로 적용된다.
  서비스에 쓰려면 그 판단이 선행돼야 한다
- **영어 서술이다.** `포근한 = cozy` 라는 번역 가정이 한 단계 들어간다.
  일반 번역이라 향 도메인 추론보다 약한 가정이지만 가정은 가정이다
- **커버리지가 {n_summary / n_total:.1%} 이고 인기 향수에 쏠려 있다** (위 표)
- **부분 문자열 검색이다.** `light` 가 `lighthearted` 에 걸리는 잡음을 제거하지 않았다
- **투표수를 쓰지 않았다.** 서술문의 존재 여부만 봤다. 동의 수를 가중하면 결과가 달라질 수 있다
- **6개 표현은 이 노트북 작성 전에 결과를 봤다** (`{PREREG["known_deviation"][:60]}...`).
  영어 대응어를 바꾸지 않았고 보지 않은 표현도 전부 포함했다
- **인과가 아니다.** 배수가 높다는 것은 동반 출현이 잦다는 뜻이지
  그 accord 가 그 표현의 원인이라는 뜻이 아니다

## 재현 방법

```bash
cd EDA
export PYTHONIOENCODING=utf-8
# 30_ai_summary_expression_accord.ipynb 를 REPORT_ONLY=False 로 실행
# API 호출 0회. 재현 게이트(accord 92개 perfume_count)가 먼저 통과해야 진행된다
```

## 관련 자료

- `SCHEMA.md` — `ai_summary` 필드 정의
- `data/scent_knowledge/domain_lexicon_v1.csv` — 대조 대상
- `docs/nlr_engineering_notes.md` 9번 — 전문가 조사 (개별 표현 근거 0건)
- `docs/spec.md` §8 남은 작업 5번 — 질감층. `무거운`·`가벼운` 을 여기서 측정했다
"""

write_output(OUTPUT_PATHS["lift"],
             lambda p: lift_df.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["crosscheck"],
             lambda p: cross_df.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["summary"],
             lambda p: p.write_text(report, encoding="utf-8"))

if not REPORT_ONLY:
    reread = pd.read_csv(OUTPUT_PATHS["crosscheck"])
    if len(reread) != len(cross_df):
        raise ValueError("저장 결과 행 수 불일치")
    print(f"검증 통과 — lift {len(lift_df)}행 / crosscheck {len(reread)}행")

저장: analysis_outputs\30_expression_accord_lift.csv
저장: analysis_outputs\30_lexicon_crosscheck.csv
저장: analysis_outputs\30_ai_summary_signal_report.md
검증 통과 — lift 781행 / crosscheck 28행


## 10. 가드 검증

In [12]:
input_hashes_after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in input_hashes_before
           if input_hashes_before[k] != input_hashes_after[k]]
if changed:
    raise RuntimeError(f"입력이 변경됐습니다: {changed}")
print("입력 해시 동일 (perfumes.jsonl 포함)")
for path in sorted(PROTECTED):
    print(f"보호 대상 미변경 확인: {path.name}  {'존재' if path.is_file() else '없음'}")
print("생성한 출력:")
for label, path in OUTPUT_PATHS.items():
    mark = "" if path.is_file() else "  (REPORT_ONLY로 미생성)"
    print(f"  {label}: {path.relative_to(PROJECT_ROOT)}{mark}")

입력 해시 동일 (perfumes.jsonl 포함)
보호 대상 미변경 확인: community_product_alias_v1.csv  존재
보호 대상 미변경 확인: domain_lexicon_v1.csv  존재
보호 대상 미변경 확인: 22_pilot_human_evaluation.csv  존재
보호 대상 미변경 확인: 13_stage1_golden_set_v1_200.xlsx  존재
보호 대상 미변경 확인: perfumes.csv  존재
보호 대상 미변경 확인: perfumes.jsonl  존재
생성한 출력:
  lift: analysis_outputs\30_expression_accord_lift.csv
  crosscheck: analysis_outputs\30_lexicon_crosscheck.csv
  summary: analysis_outputs\30_ai_summary_signal_report.md
